# TC_03: FCNN clean baseline (Golden Baseline)
Full diagnostic pipeline on clean data using RF model.
UQ: Deep Ensemble, MC Dropout
XAI: SHAP + LIME

In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.append('..')
os.chdir('..')
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

from src.utils.config_loader import load_config
from src.utils.data_loader import load_tabular_data
from src.data_diagnostics.quality_checks import check_tabular_quality
from src.utils.visualization import plot_class_distribution, plot_correlation, plot_feature_boxplots
from src.utils.performance_tracker import PerformanceTracker
from src.models.tabular_models import train_fcnn, evaluate_model_fcnn
from src.inference_diagnostics.uncertainty import deep_ensemble_prediction_tab, mc_dropout_prediction_tab
from src.inference_diagnostics.explainability import shap_tab, lime_tab,calculate_consistency_tabular
from src.inference_diagnostics.flagging import assign_flags,evaluate_flags
from src.utils.visualization import plot_uncertainty_distribution, plot_flag_distribution
from src.utils.report_generator import generate_report,save_report
from src.utils.config_loader import load_config, get_tabular_config, save_threshold
from src.utils.per_sample_logger import save_per_sample, build_experiment_id

config = load_config()
tracker = PerformanceTracker()

In [2]:
# Identify this run
dataset_short = get_tabular_config(config)['short_name']
model_name = 'fcnn'
test_case = 'TC03'
experiment_id = build_experiment_id(dataset_short, model_name, test_case)
print(f"Experiment: {experiment_id}")

Experiment: adult_fcnn_TC03


### Load data and EDA


In [3]:
X_train, X_test, y_train, y_test = load_tabular_data(config)
report = check_tabular_quality(X_train, y_train, config)
plot_class_distribution(report, f"RF {get_tabular_config(config)['name']} Baseline", config)
plot_correlation(report, f"RF {get_tabular_config(config)['name']} Baseline", config)
plot_feature_boxplots(X_train, f"RF {get_tabular_config(config)['name']} Baseline", config)

Loading dataset.
Dataset: UCI Adult Income
Duplicate rows kept (6374 found).
Found 6465 missing values. Filling with mode.
Data loaded for 39073 train, 9769 test
 Features: 12
EDA started for tabular data.
Samples: 39073, Features: 12
Class distribution: {0: 29724, 1: 9349}
Imbalance ratio: 0.3145
Duplicate rows: 5422
Total outlier count: 5915
EDA completed for UCI Adult Income
Saved: results/figures\class_distribution_rf_uci_adult_income_baseline.png
Saved: results/figures\correlation_rf_uci_adult_income_baseline.png
Saved: results/figures\boxplot_rf_uci_adult_income_baseline.png


### Train FCNN baseline

In [4]:
tracker.start_performance_track()
fcnn_model = train_fcnn(X_train, y_train, config)
tracker.stop_performance_track("FCNN training")

tracker.start_performance_track()
fcnn_accuracy, fcnn_prediction, fcnn_report = evaluate_model_fcnn(fcnn_model, X_test, y_test, config)
tracker.stop_performance_track("FCNN evaluation")

FCNN training started.
Epoch 10/50
Epoch 20/50
Epoch 30/50
Epoch 40/50
Epoch 50/50
FCNN training completed.
FCNN training: Time:21.18s, Memory:534.53MB
Model evaluation started for FCNN.
{'0': {'precision': 0.8898558254318742, 'recall': 0.9219485937289732, 'f1-score': 0.9056179775280899, 'support': 7431.0}, '1': {'precision': 0.7198067632850241, 'recall': 0.6372968349016254, 'f1-score': 0.6760435571687841, 'support': 2338.0}, 'accuracy': 0.8538233186610708, 'macro avg': {'precision': 0.8048312943584492, 'recall': 0.7796227143152993, 'f1-score': 0.7908307673484369, 'support': 9769.0}, 'weighted avg': {'precision': 0.8491582404897783, 'recall': 0.8538233186610708, 'f1-score': 0.8506742786029127, 'support': 9769.0}}
FCNN evaluation: Time:0.02s, Memory:8.77MB


{'operation': 'FCNN evaluation', 'time_seconds': 0.02, 'memory_usage': 8.77}

### Uncertainty Quantification (MC Dropout + Deep Ensemble)

In [5]:
# MC Dropout
tracker.start_performance_track()
mc_means_probabilities, mc_uncertainty = mc_dropout_prediction_tab(fcnn_model, X_test, config)
tracker.stop_performance_track("FCNN MC Dropout")

mc_threshold = round(float(np.percentile(mc_uncertainty, 90)), 4)
save_threshold(config, dataset_short, model_name, 'mc', mc_threshold)
print(f"MC Dropout uncertainty status:")
print(f"Mean: {mc_uncertainty.mean()}")
print(f"Max: {mc_uncertainty.max()}")
print(f" 90th percentile (threshold): {mc_threshold}")

plot_uncertainty_distribution(mc_uncertainty, "FCNN MC Dropout", mc_threshold, config)

MC Dropout started.
Pass 10/50 done.
Pass 20/50 done.
Pass 30/50 done.
Pass 40/50 done.
Pass 50/50 done.
MC Dropout finished for tabular.
FCNN MC Dropout: Time:0.06s, Memory:5.02MB
Threshold saved: adult_fcnn_mc = 0.0874
MC Dropout uncertainty status:
Mean: 0.04354017600417137
Max: 0.2761109173297882
 90th percentile (threshold): 0.0874
Saved: results/figures\uncertainty_distribution_fcnn_mc_dropout.png


In [6]:
# Deep Ensemble
tracker.start_performance_track()
de_means_probabilities, de_uncertainty = deep_ensemble_prediction_tab(train_fcnn, X_train, y_train, X_test, config)
tracker.stop_performance_track("FCNN Deep Ensemble prediction")

de_threshold = round(float(np.percentile(de_uncertainty, 90)), 4)
save_threshold(config, dataset_short, model_name, 'de', de_threshold)
print(f"Deep Ensembl uncertainty status:")
print(f"Mean: {de_uncertainty.mean()}")
print(f"Max: {de_uncertainty.max()}")
print(f" 90th percentile (threshold): {de_threshold}")

plot_uncertainty_distribution(de_uncertainty, "FCNN Deep Ensemble", de_threshold, config)

Deep Ensemble started for tabular.
Training model with seed 0
FCNN training started.
Epoch 10/50
Epoch 20/50
Epoch 30/50
Epoch 40/50
 Early stopping at epoch 47
FCNN training completed.
Training model with seed 1
FCNN training started.
Epoch 10/50
Epoch 20/50
Epoch 30/50
Epoch 40/50
Epoch 50/50
FCNN training completed.
Training model with seed 2
FCNN training started.
Epoch 10/50
Epoch 20/50
Epoch 30/50
Epoch 40/50
Epoch 50/50
FCNN training completed.
Training model with seed 3
FCNN training started.
Epoch 10/50
Epoch 20/50
Epoch 30/50
 Early stopping at epoch 35
FCNN training completed.
Training model with seed 4
FCNN training started.
Epoch 10/50
Epoch 20/50
Epoch 30/50
 Early stopping at epoch 40
FCNN training completed.
Deep Ensemble finished for tabular.
FCNN Deep Ensemble prediction: Time:104.45s, Memory:0.00MB
Threshold saved: adult_fcnn_de = 0.0364
Deep Ensembl uncertainty status:
Mean: 0.016269026324152946
Max: 0.18223530054092407
 90th percentile (threshold): 0.0364
Saved: re

### Explainability (SHAP and LIME)

In [7]:
# SHAP
tracker.start_performance_track()
shap_values, shap_samples = shap_tab(fcnn_model, X_train, X_test, config, is_pytorch = True)
tracker.stop_performance_track("FCNN SHAP")
print(f"SHAP values shape: {shap_values.shape}")

SHAP started.


  0%|          | 0/200 [00:00<?, ?it/s]

SHAP finished. Explained: 200 samples.
FCNN SHAP: Time:6.36s, Memory:12.63MB
SHAP values shape: (200, 12, 2)


In [8]:
# LIME
tracker.start_performance_track()
lime_explanations, lime_samples = lime_tab(fcnn_model, X_train, X_test, config, is_pytorch = True)
tracker.stop_performance_track("FCNN LIME")
print(f"LIME explanations: {len(lime_explanations)}")

LIME started.
Explained 50 / 200 samples.
Explained 100 / 200 samples.
Explained 150 / 200 samples.
Explained 200 / 200 samples.
LIME finished. Explained: 200 samples.
FCNN LIME: Time:1.86s, Memory:0.00MB
LIME explanations: 200


In [9]:
# Calculate consistency
feature_names = list(X_train.columns)
consistency_scores = calculate_consistency_tabular(shap_values, lime_explanations, feature_names, top=5)
print(f"Mean consistency: {consistency_scores.mean()}")
print(f"Min consistency: {consistency_scores.min()}")
print(f"Max consistency: {consistency_scores.max()}")


Calculate consistency tabular.
Mean consistency: 0.6169999999999998
Min consistency: 0.2
Max consistency: 0.8


### Flagging (MC Dropout + Consistency vs Deep Ensembles + Consistency)

In [10]:
# MC Dropout flags
mc_y_predictions = mc_means_probabilities[:len(consistency_scores)].argmax(axis = 1)
mc_flags = assign_flags(mc_uncertainty[:len(consistency_scores)], consistency_scores, mc_threshold, 0.5)

print(f"MC Dropout + XAI Flagging:")
mc_flag_results = evaluate_flags(mc_flags, mc_y_predictions, y_test[:len(consistency_scores)])
plot_flag_distribution(mc_flags, "FCNN MC Dropout + XAI Flagging", config)

MC Dropout + XAI Flagging:
RED: Count: 4 Accuracy:100.0%
YELLOW: Count: 37 Accuracy:78.4%
GREEN: Count: 159 Accuracy:88.7%
Flagging system is not working, needs threshold adjustment
Saved: results/figures\flagging_distribution_fcnn_mc_dropout_+_xai_flagging.png


In [11]:
# Deep Ensemble flags
de_y_predictions = de_means_probabilities[:len(consistency_scores)].argmax(axis = 1)
de_flags = assign_flags(de_uncertainty[:len(consistency_scores)], consistency_scores, de_threshold, 0.5)

print(f"Deep Ensembles + XAI Flagging:")
de_flag_results = evaluate_flags(de_flags, de_y_predictions, y_test[:len(consistency_scores)])
plot_flag_distribution(de_flags, "FCNN Deep Ensembles + XAI Flagging", config)

Deep Ensembles + XAI Flagging:
RED: Count: 2 Accuracy:100.0%
YELLOW: Count: 46 Accuracy:78.3%
GREEN: Count: 152 Accuracy:90.8%
Flagging system is not working, needs threshold adjustment
Saved: results/figures\flagging_distribution_fcnn_deep_ensembles_+_xai_flagging.png


### UQ only vs UQ + XAI comparison

In [12]:
# MC Dropout without XAI (constant consistency)
con_consistency = np.ones(len(consistency_scores))
mc_flags_uq_only = assign_flags(mc_uncertainty[:len(consistency_scores)], con_consistency, mc_threshold, 0.5)

print(f"MC Dropout without XAI Flagging:")
mc_only_flag_results = evaluate_flags(mc_flags_uq_only, mc_y_predictions, y_test[:len(consistency_scores)])
plot_flag_distribution(mc_flags_uq_only, "FCNN MC Dropout only", config)

print(f"MC Dropout with XAI green accuracy: {mc_flag_results['GREEN']['accuracy']}")
print(f"MC Dropout without XAI green accuracy: {mc_only_flag_results['GREEN']['accuracy']}")
print(f"Improvement with XAI: {mc_flag_results['GREEN']['accuracy'] - mc_only_flag_results['GREEN']['accuracy']}")


MC Dropout without XAI Flagging:
RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 20 Accuracy:70.0%
GREEN: Count: 180 Accuracy:88.9%
Flagging system is working
Saved: results/figures\flagging_distribution_fcnn_mc_dropout_only.png
MC Dropout with XAI green accuracy: 0.8868
MC Dropout without XAI green accuracy: 0.8889
Improvement with XAI: -0.0020999999999999908


In [13]:
# Deep Ensemble without XAI (constant consistency)
de_flags_uq_only = assign_flags(de_uncertainty[:len(consistency_scores)], con_consistency, de_threshold, 0.5)

print(f"Deep Ensemble without XAI Flagging:")
de_only_flag_results = evaluate_flags(de_flags_uq_only, de_y_predictions, y_test[:len(consistency_scores)])
plot_flag_distribution(de_flags_uq_only, "FCNN Deep Ensembles only", config)

print(f"Deep Ensembles with XAI green accuracy: {de_flag_results['GREEN']['accuracy']}")
print(f"Deep Ensembles without XAI green accuracy: {de_only_flag_results['GREEN']['accuracy']}")
print(f"Improvement with XAI: {de_flag_results['GREEN']['accuracy'] - de_only_flag_results['GREEN']['accuracy']}")


Deep Ensemble without XAI Flagging:
RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 25 Accuracy:68.0%
GREEN: Count: 175 Accuracy:90.9%
Flagging system is working
Saved: results/figures\flagging_distribution_fcnn_deep_ensembles_only.png
Deep Ensembles with XAI green accuracy: 0.9079
Deep Ensembles without XAI green accuracy: 0.9086
Improvement with XAI: -0.0006999999999999229


In [14]:
# Per sample results for the ablation and metrics analysis
save_per_sample(
    config,
    experiment_id,
    y_true=y_test,
    y_pred=mc_y_predictions,
    mc_uncertainty=mc_uncertainty,
    de_uncertainty=de_uncertainty,
    consistency=consistency_scores,
)

Per sample results saved: results/per_sample\adult_fcnn_TC03.csv (200 rows)


,sample_id,y_true,y_pred,correct,mc_uncertainty,de_uncertainty,consistency
0,0,0,0,1,0.010620,0.001813,0.6
1,1,0,0,1,0.058631,0.027592,0.6
2,2,0,0,1,0.003746,0.000471,0.6
3,3,0,0,1,0.043706,0.016156,0.6
4,4,1,1,1,0.068877,0.048996,0.6
...,...,...,...,...,...,...,...
195,195,0,0,1,0.039535,0.003921,0.8
196,196,0,0,1,0.068188,0.058026,0.6
197,197,0,0,1,0.018505,0.000859,0.6
198,198,1,0,0,0.110468,0.016884,0.6


### Save golden baseline report

In [15]:
fcnn_baseline = {
    'model': 'FCNN',
    'accuracy': round(fcnn_accuracy, 4),
    'classification_report': fcnn_report,
    'mc_uncertainty': {
        'mean': round(float(mc_uncertainty.mean()), 8),
        'max': round(float(mc_uncertainty.max()), 8),
        'threshold': mc_threshold
    },
    'de_uncertainty': {
        'mean': round(float(de_uncertainty.mean()), 8),
        'max': round(float(de_uncertainty.max()), 8),
        'threshold': de_threshold
    },
    'consistency':{
        'mean': round(float(consistency_scores.mean()), 4),
        'min': round(float(consistency_scores.min()), 4),
        'max': round(float(consistency_scores.max()), 4)
    },
    'flagging_mc': mc_flag_results,
    'flagging_de': de_flag_results,
    'flagging_mc_only': mc_only_flag_results,
    'mc_vs_mc_plus_xai': round(mc_flag_results['GREEN']['accuracy'] - mc_only_flag_results['GREEN']['accuracy'], 4),
    'flagging_de_only': de_only_flag_results,
    'de_vs_de_plus_xai': round(de_flag_results['GREEN']['accuracy'] - de_only_flag_results['GREEN']['accuracy'], 4),
    'performance': tracker.get_results()
}

report_output = generate_report(config, f"{get_tabular_config(config)['name']} - FCNN clean baseline", stage1_result=fcnn_baseline)
save_report(report_output, f'{dataset_short.upper()}_TC_03_FCNN_Golden_Baseline.json', config)

Generating report.
Diagnostic report generated.
Saving report.
